# Organic history

This notebook runs an organic culture history on a generated planet: peoples start as foragers, take up farming, raise their energy, transport and communication, form states, fight, colonise, mix and meet the planet's eruptions, droughts and pandemics. It then reads the chronicle, plots the history over time, draws maps at the kept epochs and zooms into a region as it stood in the past.

Run it from the `notebooks/` folder after `pip install -e ".[dev,notebooks]"`. The notebook runs in about a minute on the preview grid; the standard grid and a full Holocene take a few minutes.

In [ ]:
%matplotlib inline
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image

from worldgen import generate_world, load_spec, load_world, save_world
from worldgen.culture import CultureSpec, apply_culture, format_chronicle, world_at_epoch
from worldgen.render import animate_epochs, plot_chronicle, plot_epochs, plot_local_map, plot_map
from worldgen.report import format_culture

## 1. A planet

The Earth example on the preview grid (10k cells), with the fast heuristic plate layout.

In [ ]:
spec = load_spec("../examples/earth.yaml")
spec = spec.model_copy(update={"surface": spec.surface.model_copy(update={"tectonics": "heuristic"})})
world = generate_world(spec, resolution="preview")
print(world.state.name, f"land {1 - world.surface['ocean'].values.mean():.0%}")

## 2. An organic culture spec

`mode: organic` makes a culture's development, people and states emerge from the history instead of being given. The `history` block sets the span (default 12 000 years), the epochs to keep, and events placed by hand. The planet's own eruptions, droughts, pandemics and impacts are drawn at their rates unless `planet_events` is false. An organic colony lands at its date with its given development.

In [ ]:
s = world.surface
land, lat, lon = ~s["ocean"].values, s.lat.values, s.lon.values
south = np.flatnonzero(land & (lat < -10))                     # a homeland for Bel in the south
temperate = np.flatnonzero(land & (np.abs(lat - 40) < 8))      # a landing site for Corin
bel, site = south[south.size // 2], temperate[temperate.size // 3]
place = lambda cell: {"lat_deg": float(lat[cell]), "lon_deg": float(lon[cell])}

culture = CultureSpec.model_validate({
    "history": {
        "start_years_ago": 8000,
        "epochs": [6000, 4000, 2000, 1000, 500],
        "events": [
            {"kind": "drought", "years_ago": 4200, **place(site), "size": 0.4, "radius_km": 1500,
             "duration_years": 250},
            {"kind": "eruption", "years_ago": 1500, "size": 50},
        ],
    },
    "cultures": [
        {"name": "Aru", "mode": "organic"},
        {"name": "Bel", "mode": "organic", "homeland": {**place(bel), "radius_km": 3000}},
        {"name": "Corin", "mode": "organic", "inhabitants": "colonist",
         "development": {"energy": "water_wind", "transport": "wheel_and_sail", "communication": "print"},
         "colony": {"founded_years_ago": 700, "landing_sites": [place(site)]}},
    ],
})
world = apply_culture(world, culture)
civ = world.culture
print(format_culture(civ, world))

The report ends with the history table: each kept epoch's people, cultures, states, largest state and the highest levels reached. Cultures that arose during the history (overseas colonies of the natives' states, creoles) appear with their own names.

## 3. The chronicle

`format_chronicle` writes major entries in full and counts minor ones (secessions, conquests, small wars) per period. It filters by time, kind and culture; state and culture numbers are replaced by names where a kept epoch named them. `worldgen chronicle <world>` prints the same from the command line.

In [ ]:
print(format_chronicle(civ, since=2000, until=500))

In [ ]:
print(format_chronicle(civ, kinds=["independence", "creole", "pandemic", "eruption", "drought"], minor="hide"))

## 4. The history over time

The history records each culture's people, land, states and mean levels at every step (`civ.series`). `plot_chronicle` draws people and mean energy per culture with the chronicle's major entries below; dotted lines mark the kept epochs.

In [ ]:
fig = plot_chronicle(world)

In [ ]:
s = civ.series
for c in civ.cultures:
    mine = s.culture == c.id
    if mine.any():
        k = np.argmax(s.people[mine])
        print(f"{c.name or c.id:<14} peak {s.people[mine][k]:>14,.0f} people {s.years_ago[mine][k]:>6,.0f} years ago, "
              f"largest state {s.largest_state_km2[mine].max() / 1e6:5.2f} M km²")

## 5. Maps at the kept epochs

`plot_epochs` draws one culture field at each kept epoch and `animate_epochs` writes them as a GIF. A state or culture keeps its colour from epoch to epoch. `world_at_epoch` returns the world as it stood at one epoch, for any map or report; on the command line, `--epoch` does the same for `map`, `report`, `zoom` and `culture`.

In [ ]:
fig = plot_epochs(world, "political")

In [ ]:
gif = animate_epochs(world, Path(tempfile.mkdtemp()) / "epochs.gif", "cultures", fps=1, width=600)
Image(filename=gif)

In [ ]:
past = world_at_epoch(world, 2000)
ax = plot_map(past, "population", "robinson", title="")

## 6. A region in the past

Zooming works on any epoch: the region's settlements, roads and borders come from the civilisation at that date.

In [ ]:
from worldgen.culture.local import zoom_culture
from worldgen.zoom import zoom_region

st = past.culture.settlements
lat, lon = float(st.lat_deg[np.argmax(st.population)]), float(st.lon_deg[np.argmax(st.population)])
region = zoom_region(past, lat, lon, half_width_km=60)
fig = plot_local_map(past, region, culture=zoom_culture(past, region), title="")

## 7. Saving

A saved world keeps the civilisation, its kept epochs (under `culture/epochs/`), the chronicle and the series.

In [ ]:
folder = save_world(world, Path(tempfile.mkdtemp()) / "organic")
again = load_world(folder).culture
print([e.epoch_years_ago for e in again.epochs], len(again.chronicle), len(again.series))